1. 문서의 내용을 읽는다
2. 문서를 쪼갠다
    - 토큰수 초과로 답변을 생성하지 못할 수 있고
    - 문서가 길면 답변 생성이 오래걸림
3. 임베딩 --> 벡터 데이터 베이스에 저장
4. 질문이 있을 때, 벡터 데이터베이스에 유사도 검색
5. 유사도 검색으로 가져온 문서를 LLM에 질문과 같이 전달

python.langchain 들어가면 docs 변화 방법 있음

In [1]:
from langchain_community.document_loaders import Docx2txtLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1500,
    chunk_overlap=200,
)

loader = Docx2txtLoader('./tax.docx')
document_list = loader.load_and_split(text_splitter)

In [2]:
from dotenv import load_dotenv
from langchain_upstage import UpstageEmbeddings

load_dotenv()

embedding = UpstageEmbeddings(model='embedding-passage')

In [3]:
from langchain_chroma import Chroma

#database =Chroma.from_documents(documents=document_list, embedding=embedding,collection_name='chroma-tax' ,persist_directory="./chroma")
database =Chroma(collection_name='chroma-tax' ,persist_directory="./chroma",embedding_function=embedding)


In [4]:
query ='연봉 5천만원인 직장인의 소득세는 얼마인가요?'
#retrieved_docs = database.similarity_search(query, k=3)

In [5]:
from langchain_upstage import ChatUpstage

llm = ChatUpstage()

In [6]:
# prompt = f"""[Identity]
# - 당신은 최고의 한국 소득세 전문가 입니다
# - [Context]를 참고해서 사용자의 질문에 답변해주세요

# [Context]
# {retrieved_docs}

# Question: {query}
# """

In [7]:
# ai_message = llm.invoke(prompt)

In [8]:
# ai_message.content

In [9]:
from langchain_classic import hub

prompt = hub.pull("rlm/rag-prompt")

In [10]:
from langchain_classic.chains import RetrievalQA

qa_chain = RetrievalQA.from_chain_type(
    llm,
    retriever=database.as_retriever(), 
    chain_type_kwargs={"prompt":prompt},
)

In [11]:
ai_message = qa_chain({"query":query})

C:\Users\JJH\AppData\Local\Temp\ipykernel_42676\980961933.py:1: LangChainDeprecationWarning: The method `Chain.__call__` was deprecated in langchain-classic 0.1.0 and will be removed in 1.0. Use `invoke` instead.
  ai_message = qa_chain({"query":query})


In [12]:
ai_message

{'query': '연봉 5천만원인 직장인의 소득세는 얼마인가요?',
 'result': '연봉 5천만원인 직장인의 소득세는 정확한 공제액에 따라 달라질 수 있습니다. 그러나 제공된 정보에 따르면, 근로소득세액공제는 총급여액이 3천 300만원 초과 7천만원 이하인 경우 66만원부터 50만원 범위 내에 있습니다. 자녀세액공제는 자녀가 없는 경우 공제액이 없으며, 자녀가 2명인 경우 연 55만원이 공제됩니다. 따라서 정확한 소득세는 개인의 상황에 따라 다를 수 있으니, 정확한 계산은 국세청 홈택스 등을 통해 확인하시는 것이 좋습니다.'}